### How to Use `__slots__` Like a Pro

### 1. Basic usage

In [1]:
class Point:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

In [2]:
p = Point(3, 3)
Point.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              '__slots__': ('x', 'y'),
              '__init__': <function __main__.Point.__init__(self, x, y)>,
              '__static_attributes__': ('x', 'y'),
              'x': <member 'x' of 'Point' objects>,
              'y': <member 'y' of 'Point' objects>,
              '__doc__': None})

In [3]:
p.__dict__

AttributeError: 'Point' object has no attribute '__dict__'

- No `__dict__`, no dynamic attributes.
- Saves memory (**no per‑instance dict**)
- Attribute access is faster

### 2. Allowing a `__dict__` optionally

In [4]:
class Flexible:
    __slots__ = ("name", "__dict__")

In [6]:
f = Flexible()
f.name = "Denys"      # stored in slot

In [7]:
f.name

'Denys'

In [9]:
f.age = 30            # stored in __dict__
f.country = "CZ"      # stored in __dict__

In [10]:
f.__dict__

{'age': 30, 'country': 'CZ'}

**You get the best of both worlds**.

### 3. Slots - Runtime Immutability
- No `__dict__` → **cannot bypass immutability**
- **No new attributes can be added**
- **Only `_value` and `_is_frozen` exist**
- Faster and smaller than a normal class

> This is a classic **freeze‑after‑init** pattern.

In [2]:
class Frozen:
    __slots__ = ("_value", "_is_frozen")

    def __init__(self, value):
        # bypass your overridden __setattr__
        object.__setattr__(self, "_is_frozen", False)
        object.__setattr__(self, "_value", value)
        object.__setattr__(self, "_is_frozen", True)

    @property
    def value(self):
        return self._value

    def __setattr__(self, name, value):
        if getattr(self, "_is_frozen", False):
            raise AttributeError("Object is immutable")
        object.__setattr__(self, name, value)

**1. Why __slots__ is used here**

`__slots__ = ("_value", "_is_frozen")` means:
- The instance has no `__dict__`
- Only `_value` and `_is_frozen` can exist
- Memory footprint is smaller
- Attribute access is faster
- You cannot add new attributes dynamically

This is perfect for **immutable** objects.

**2. Why `object.__setattr__` is used in `__init__`**

Inside `__init__`, you **bypass** your overridden `__setattr__`:
```py
object.__setattr__(self, "_is_frozen", False)
object.__setattr__(self, "_value", value)
object.__setattr__(self, "_is_frozen", True)
```
Why?

Because your custom `__setattr__` **blocks all assignments** once `_is_frozen` is `True`.

If you used `self._value = value`, it would call your overridden `__setattr__`, which would **immediately raise** an exception.

So you must bypass it using the base implementation.

**3. How immutability is enforced**

Your custom `__setattr__`:
```py
def __setattr__(self, name, value):
    if getattr(self, "_is_frozen", False):
        raise AttributeError("Object is immutable")
    object.__setattr__(self, name, value)
```
This means:
- During `__init__`, `_is_frozen` is `False` → assignments **allowed**
- After `__init__`, `_is_frozen` is `True` → all assignments **forbidden**

Example:

In [ ]:
f = Frozen(10)
print(f.value)   # 10

f._value = 20    # AttributeError: Object is immutable

10


AttributeError: Object is immutable

In [14]:
f.new_attr = 99  # AttributeError: Object is immutable

AttributeError: Object is immutable

### 4. Example usage

In [3]:
f = Frozen(42)
print(f.value)  # 42

try:
    f._value = 100
except AttributeError as e:
    print(e)

42
Object is immutable
